# ESM-2 Protein Localization: Project Walkthrough

## Objective

Predict whether a protein is **membrane-associated (`1`)** or **soluble (`0`)** from its amino-acid sequence, while comparing simple biological features with frozen protein-language-model representations.

The project uses the resource-efficient `facebook/esm2_t6_8M_UR50D` checkpoint. ESM-2 remains frozen in the completed baseline; end-to-end fine-tuning is the next experiment.

## Experimental progression

```text
Protein sequence
├── Composition + hydrophobicity features
│   ├── Logistic Regression
│   └── Random Forest
│
└── Frozen ESM-2 8M
    ├── First-token representation
    ├── Mean-pooled residue representation
    └── Max-pooled residue representation
        ↓
    Logistic Regression
```

The biological baselines test how much can be learned from explicit hydrophobicity and composition. The frozen ESM-2 experiment tests whether pretrained contextual representations add useful information.

## Data and splits

The ESM-compatible experiment contains 7,890 DeepLoc proteins after retaining membrane and soluble labels, removing exact duplicate development sequences, and excluding 737 sequences longer than 1,022 residues.

| Split | Soluble | Membrane | Total | Purpose |
|---|---:|---:|---:|---|
| Train | 2,957 | 2,098 | 5,055 | Fit model parameters |
| Validation | 739 | 525 | 1,264 | Compare representations and checkpoints |
| Test | 905 | 666 | 1,571 | Exploratory evaluation |
| **Total** | **4,601** | **3,289** | **7,890** | |

The full-length hydrophobicity experiment separately retains 8,627 proteins, including all 737 sequences above the ESM-2 length limit. Because the evaluated protein populations differ, its full-length score is not an exact apples-to-apples comparison with ESM-2.

## Methods

### Biological features

Thirty-five interpretable features summarize amino-acid composition, sequence length, Kyte–Doolittle hydropathy, charged and aromatic fractions, hydrophobic runs, and sliding-window hydropathy.

### Frozen ESM-2

Each residue receives a 320-dimensional contextual representation. Three fixed-size protein representations are derived in one forward pass:

- first special token;
- mean of residue embeddings, excluding special tokens and padding; and
- dimension-wise maximum of residue embeddings.

A class-balanced Logistic Regression pipeline with `StandardScaler` is fitted independently to each representation.

## Frozen ESM-2 validation results

| Representation | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| **Mean pooling** | **0.8916** | **0.8819** | **0.8533** | **0.8674** | **0.9479** |
| First token | 0.8513 | 0.8197 | 0.8229 | 0.8213 | 0.9348 |
| Max pooling | 0.8339 | 0.8046 | 0.7924 | 0.7985 | 0.9114 |

Mean pooling ranks first on the predefined primary metric, validation F1.

In [ ]:
from pathlib import Path
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == "notebooks" else current
metrics_path = PROJECT_ROOT / "results/metrics/embedding_classifiers/validation_metrics.csv"
if metrics_path.exists():
    display(pd.read_csv(metrics_path).sort_values("f1", ascending=False).style.format({
        metric: "{:.4f}" for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]
    }))
else:
    print("Saved metrics are unavailable. Run scripts/train_embedding_classifiers.py first.")

## Exploratory test results

| Representation | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| **Mean pooling** | **0.8765** | **0.8512** | **0.8589** | **0.8550** | **0.9426** |
| First token | 0.8568 | 0.8257 | 0.8393 | 0.8325 | 0.9296 |
| Max pooling | 0.8141 | 0.7664 | 0.8078 | 0.7865 | 0.8995 |

> All three test results were inspected for learning and comparison. The official test split therefore participated in exploratory judgment and should not be described as fully untouched evidence for final model selection.

## Full-length biological baselines

| Model | Validation accuracy | Validation F1 | Test accuracy | Test F1 |
|---|---:|---:|---:|---:|
| Hydrophobicity + Logistic Regression | 0.8154 | 0.7682 | 0.8136 | 0.7791 |
| **Hydrophobicity + Random Forest** | **0.8467** | **0.7917** | **0.8433** | **0.8009** |

Random Forest improves the same 35 biological features through nonlinear thresholds and interactions. It raises precision while slightly reducing membrane recall relative to Logistic Regression.

## Validation ROC comparison for biological baselines

The saved figure compares ranking performance across all classification thresholds. Random Forest reaches validation ROC-AUC `0.92`, compared with `0.87` for Logistic Regression.


In [ ]:
from IPython.display import Image, display
figure_path = PROJECT_ROOT / "results/figures/validation_roc_hydrophobicity_classifiers.png"
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    print("Figure unavailable. Reproduce it with the public baseline workflow or scripts.")

## Interpretation

- Hydrophobicity and composition already provide substantial localization signal.
- Random Forest shows that nonlinear interactions improve those engineered features.
- Frozen mean-pooled ESM-2 performs better on its ESM-compatible evaluation population, suggesting that contextual sequence representations contain useful information beyond simple hydrophobicity.
- PCA and UMAP do not show two perfectly isolated label clusters. Strong linear-probe performance indicates that useful localization information is distributed across the full 320-dimensional embedding space.

## Reproduction outline

After obtaining the DeepLoc FASTA according to its license and placing it at `data/raw/deeploc_data.fasta`:

```bash
python scripts/prepare_data.py
python scripts/extract_embeddings.py --device auto
python scripts/train_embedding_classifiers.py
```

Fine-tuning uses the public Colab notebook `02_colab_finetuning.ipynb`, which calls:

```bash
python scripts/train_finetune.py --device cuda --mixed-precision auto
```

Large datasets, cached embeddings, and trained model artifacts are intentionally excluded from Git.

## Limitations

- ESM-2 direct inputs are limited to 1,022 residues in this workflow.
- Exact duplicates are handled, but homologous proteins may still cross splits.
- Dataset source, license, download date, and label provenance must be finalized in the repository documentation.
- The exploratory test split has been inspected for multiple representations.
- Computational localization predictions do not replace experimental validation.
- Fine-tuning results are not yet available.

## Next experiment

Fine-tune ESM-2 8M with residue mean pooling, class-weighted cross-entropy, mixed precision, gradient accumulation, early stopping, and validation-F1 checkpoint selection. Compare it against frozen mean-pooled ESM-2 using the same ESM-compatible proteins and metrics.